# First Notebook to use

This notebook scrapes the page letterboxd for my own personal rating of movies, plus the homepage of the reviewed movies.

I have reviewed the https://letterboxd.com/robots.txt file, and those pages are all allowed.

In [40]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import numpy as np
import time

In [41]:
BASE_URL = "https://letterboxd.com"
PROFILE_URL = "https://letterboxd.com/bronzemarkian/films/"

headers = {"User-Agent": "Mozilla/5.0"}

# Step 1: Get all movie links from your profile page
r = requests.get(PROFILE_URL, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")
r.status_code # 200 means everything okay

200

In [42]:
# Loop over each available page

page = 1
all_movies = []
while True:
    url = f'{PROFILE_URL}page/{page}/'
    resp = requests.get(url, headers=headers)
    if resp.status_code != 200:
        break
    soup = BeautifulSoup(resp.text, "html.parser")
    movies = soup.find_all('li', class_='griditem') # from inspecting page, 'griditem' is the class holding info about each movie
    if not movies: # this is what breaks when it tries page 3 and there are no movies there
        break
    all_movies += movies
    page += 1


In [43]:
def stars_to_number(stars): # to get actual rating numbers
    if stars:
        num_stars = stars.count('★')
        half_star = stars.count('½')
        return num_stars + 0.5*half_star
    else:
        return None
    
def find_features_from_link(movie_link):
    resp = requests.get(movie_link, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")
    avg_rating = float(soup.find_all("meta", attrs={"name": "twitter:data2"})[0]["content"].split(' ')[0])
    movie_genres = [a.text.strip() for a in soup.select('a.text-slug[href^="/films/genre/"]')]
    movie_director = soup.find_all("meta", attrs={"name": "twitter:data1"})[0]["content"]
    movie_countries = [a.text.strip() for a in soup.select('a.text-slug[href^="/films/country/"]')]  
    movie_languages = [a.text.strip() for a in soup.select('a.text-slug[href^="/films/language/"]')]

    return avg_rating, movie_genres, movie_director, movie_countries, movie_languages

    
titles = []
years = []
personal_ratings = []
links = []

for movie in all_movies:
    # The title and link are in the 'data-item-name' and 'data-item-link' attributes of the div
    div = movie.find('div', class_='react-component')
    title_year = div['data-item-name']
    movie_link = "https://letterboxd.com" + div['data-item-link']
    
    rating_tag = movie.find('span', class_='rating')
    personal_movie_rating = rating_tag.text if rating_tag else None

    movie_title = title_year.split("(")[0][:-1]
    movie_year = int(title_year.split("(")[1].replace(")", ""))
    
    titles.append(movie_title)
    years.append(movie_year)
    personal_ratings.append(stars_to_number(personal_movie_rating))
    links.append(movie_link)
    # print(title, rating, stars_to_number(rating), link)

avg_ratings = []
genres = []
directors = []
countries = []
languages = []

for link in links:
    
    avg_rating, genre, director, country, language = find_features_from_link(link)
    avg_ratings.append(avg_rating)
    genres.append(genre)
    directors.append(director)
    countries.append(country)
    languages.append(language)
    time.sleep(1)  # dont hammer server and risk getting blocked



In [44]:
rows = []

for i in range(len(titles)):
    rows.append({
        "title": titles[i],
        "release_year": years[i],
        "personal_rating": personal_ratings[i],
        "avg_rating": avg_ratings[i],
        "genre": genres[i],
        "director": directors[i],
        "country": countries[i],
        "language": languages[i]
    })
    df = pd.DataFrame(rows)

In [45]:
import re
def seperate_directors(s):
    if not s:
        return []
    if type(s) != str:
        return s

    s = re.sub(r'\bet\s+al\.?\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r',\s*,', ',', s)
    s = s.strip(' ,')   # removes trailing whitespace and/or commas

    return [d.strip() for d in s.split(',') if d.strip()]

In [49]:
df2 = df[:]
df2['director'] = df2['director'].apply(seperate_directors)

In [50]:
# df.to_csv('initial_df.csv', index=False)
df2.to_csv('initial_dframe.csv', index=False)

Encoding below is done later so not used here.

In [12]:
import ast

cat_features = ['genre', 'country', 'language']

df = pd.read_csv('initial_df.csv')
for column in cat_features:
    df[column] = df[column].apply(lambda x: ast.literal_eval(x))

In [13]:
# Now the dataframe is mostly done, but there are a lot of categorical features, with multiple inputs too. I want to encode them.

from sklearn.preprocessing import MultiLabelBinarizer

def encode_category(dframe, column_name, name_of_new_column):

    mlb = MultiLabelBinarizer()
    column_encoded = mlb.fit_transform(dframe[column_name])
    new_dframe = pd.DataFrame(column_encoded, columns=[f"{column_name}: {g}" for g in mlb.classes_], index=dframe.index)
    # columns=[f"Genre: {g}" for g in mlb.classes_], 
    new_dframe[name_of_new_column] = 0

    for category in new_dframe.columns:
        if new_dframe[category].sum() < 2: # if rare category
            new_dframe[name_of_new_column] = new_dframe[name_of_new_column] | new_dframe[category] # 1 in same row if 1 already in new column or 1 in category, else 0
            new_dframe = new_dframe.drop(category, axis=1) # drop the rare category
    
    dframe = dframe.drop(column_name, axis=1)

    return pd.concat([dframe, new_dframe], axis=1)


In [11]:
print(type(df['genre'].iloc[0]))

<class 'list'>


In [16]:
df_encoded = df[:]
for column in cat_features:
    df_encoded = encode_category(df_encoded, column, f'rare {column}')
df_encoded.head(10)

,title,release_year,personal_rating,avg_rating,director,genre: Action,genre: Adventure,genre: Animation,genre: Comedy,genre: Crime,...,language: Italian,language: Japanese,language: Latin,language: Portuguese,language: Russian,language: Spanish,language: Swedish,language: Urdu,language: Xhosa,rare language
0,Oppenheimer,2023,4.5,4.17,Christopher Nolan,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,Spider-Man: Across the Spider-Verse,2023,5.0,4.41,"Kemp Powers, Justin K. Thompson et al",1,1,1,0,0,...,1,0,0,0,0,1,0,0,0,0
2,The Super Mario Bros. Movie,2023,4.0,3.22,"Aaron Horvath, Michael Jelenic",0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,Avatar: The Way of Water,2022,3.0,3.62,James Cameron,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Jujutsu Kaisen 0,2021,4.0,3.93,Sunghoo Park,1,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
5,Spider-Man: No Way Home,2021,4.0,3.83,Jon Watts,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,1
6,No Time to Die,2021,3.0,3.52,Cary Joji Fukunaga,1,1,0,0,0,...,1,0,0,0,1,1,0,0,0,0
7,Dune,2021,3.5,3.88,Denis Villeneuve,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,JUJUTSU KAISEN,2020,3.5,4.22,"Yui Umemoto, Ryohei Takeshita et al",0,0,1,0,0,...,0,1,0,0,0,0,0,0,0,0
9,Another Round,2020,4.0,4.09,Thomas Vinterberg,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0


# Below this is testing

In [161]:
# Now the dataframe is mostly done, but there are a lot of categorical features, with multiple inputs too. I want to encode them.

from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df['genres'])

genre_df = pd.DataFrame(genre_encoded, columns=[f"Genre: {g}" for g in mlb.classes_], index=df.index)

genre_df['New Genre'] = 0

for genre in genre_df.columns:
    if genre_df[genre].sum() < 2: # rare genre, so want a 1 in new genre in the same row
        print(genre)
        genre_df['New Genre'] = genre_df['New Genre'] | genre_df[genre] # 1 in same row is 1 already in new genre column or genre column, else 0
        genre_df = genre_df.drop(genre, axis = 1)

genre_df.head()

Genre: Music
Genre: Western


,Genre: Action,Genre: Adventure,Genre: Animation,Genre: Comedy,Genre: Crime,Genre: Drama,Genre: Family,Genre: Fantasy,Genre: History,Genre: Horror,Genre: Mystery,Genre: Romance,Genre: Science Fiction,Genre: Thriller,Genre: War,New Genre
0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0
1,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0
2,0,1,1,1,0,0,1,1,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0


In [154]:
for genre in genre_df.columns:
    print(genre_df[genre].sum(), genre)

58 Action
58 Adventure
17 Animation
17 Comedy
16 Crime
38 Drama
15 Family
21 Fantasy
2 History
4 Horror
2 Mystery
3 Romance
43 Science Fiction
26 Thriller
3 War
2 New Genre


In [141]:
genre_df['New Genre'].sum()

np.int64(0)

In [ ]:
genre_df.columns[0]

'Action'

In [4]:
def stars_to_number(stars): # to get actual rating numbers
    if stars:
        num_stars = stars.count('★')
        half_star = stars.count('½')
        return num_stars + 0.5*half_star
    else:
        return None

In [89]:
titles = []
ratings = []
links = []

for movie in all_movies:
    # The title and link are in the 'data-item-name' and 'data-item-link' attributes of the div
    div = movie.find('div', class_='react-component')
    title_year = div['data-item-name']
    link = "https://letterboxd.com" + div['data-item-link']
    
    rating_tag = movie.find('span', class_='rating')
    rating = rating_tag.text if rating_tag else None

    title = title_year.split("(")[0][:-1]
    year = int(title_year.split("(")[1].replace(")", ""))

    titles.append(title)
    ratings.append(rating)
    links.append(link)
    # print(title, rating, stars_to_number(rating), link)
    print(title)

Oppenheimer
Spider-Man: Across the Spider-Verse
The Super Mario Bros. Movie
Avatar: The Way of Water
Jujutsu Kaisen 0
Spider-Man: No Way Home
No Time to Die
Dune
JUJUTSU KAISEN
Another Round
Tenet
Joker
Avengers: Endgame
Checkered Ninja
Spider-Man: Into the Spider-Verse
The Meg
Avengers: Infinity War
Star Wars: The Last Jedi
Blade Runner 2049
Dunkirk
Logan
Moana
Warcraft
The Conjuring 2
Deadpool
Star Wars: The Force Awakens
Spectre
The Martian
Avengers: Age of Ultron
Interstellar
Transformers: Age of Extinction
Edge of Tomorrow
Boyhood
Whiplash
The Wolf of Wall Street
Frozen
The Conjuring
The Great Gatsby
Django Unchained
Skyfall
The Dark Knight Rises
The Avengers
The Intouchables
Harry Potter and the Deathly Hallows: Part 2
Transformers: Dark of the Moon
Fast Five
Harry Potter and the Deathly Hallows: Part 1
The Social Network
Inception
How to Train Your Dragon
Avatar
2012
Harry Potter and the Half-Blood Prince
Transformers: Revenge of the Fallen
Inglourious Basterds
Fast & Furious
Th

In [8]:
len(links)

116

In [25]:
links[0]

'https://letterboxd.com/film/oppenheimer-2023/genres/'

In [83]:
resp2 = requests.get(links[0], headers=headers)
soup_link = BeautifulSoup(resp2.text, "html.parser")
resp2.status_code

200

In [68]:
soup_link


<!DOCTYPE html>

<html class="context-client-unknown no-mobile no-js" id="html" lang="en">
<head>
<meta charset="utf-8"/>
<meta content="width=1024" name="viewport"/>
<meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>
<meta content="The story of J. Robert Oppenheimer's role in the development of the atomic bomb during World War II." name="description"/>
<meta content="video.movie" property="og:type"/>
<meta content="https://letterboxd.com/film/oppenheimer-2023/genres/" property="og:url"/>
<meta content="Oppenheimer (2023)" property="og:title"/>
<meta content="The story of J. Robert Oppenheimer's role in the development of the atomic bomb during World War II." property="og:description"/>
<meta content="https://a.ltrbxd.com/resized/sm/upload/mn/uu/op/02/oppenheimer-2023-1200-1200-675-675-crop-000000.jpg?v=aa1b3a6d85" property="og:image"/><meta content="1200" property="og:image:width"/><meta content="675" property="og:image:height"/>
<meta content="173683136069040" property=

In [84]:
genres = [
    a.text.strip()
    for a in soup_link.select('a.text-slug[href^="/films/genre/"]')
]
genres

['Drama', 'History']

In [42]:
soup_link.select('a.text-slug[href^="/films/genre/"]')

[<a class="text-slug" href="/films/genre/drama/">Drama</a>,
 <a class="text-slug" href="/films/genre/history/">History</a>]

In [43]:
soup_link.select('a[href$="ratings/"]')

[]

In [36]:
soup_link.find_all('a', class_="text-slug")

[<a class="text-slug tooltip" href="/actor/cillian-murphy/" title="J. Robert Oppenheimer">Cillian Murphy</a>,
 <a class="text-slug tooltip" href="/actor/emily-blunt/" title="Kitty Oppenheimer">Emily Blunt</a>,
 <a class="text-slug tooltip" href="/actor/matt-damon/" title="Leslie Groves">Matt Damon</a>,
 <a class="text-slug tooltip" href="/actor/robert-downey-jr/" title="Lewis Strauss">Robert Downey Jr.</a>,
 <a class="text-slug tooltip" href="/actor/florence-pugh/" title="Jean Tatlock">Florence Pugh</a>,
 <a class="text-slug tooltip" href="/actor/josh-hartnett/" title="Ernest Lawrence">Josh Hartnett</a>,
 <a class="text-slug tooltip" href="/actor/casey-affleck/" title="Boris Pash">Casey Affleck</a>,
 <a class="text-slug tooltip" href="/actor/rami-malek/" title="David Hill">Rami Malek</a>,
 <a class="text-slug tooltip" href="/actor/kenneth-branagh/" title="Niels Bohr">Kenneth Branagh</a>,
 <a class="text-slug tooltip" href="/actor/benny-safdie/" title="Edward Teller">Benny Safdie</a>,
 

In [35]:
soup_link.find_all('a', class_='tooltip display-rating -highlight')

[]

In [62]:
soup_link.find_all("meta")

[<meta charset="utf-8"/>,
 <meta content="width=1024" name="viewport"/>,
 <meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>,
 <meta content="The story of J. Robert Oppenheimer's role in the development of the atomic bomb during World War II." name="description"/>,
 <meta content="video.movie" property="og:type"/>,
 <meta content="https://letterboxd.com/film/oppenheimer-2023/genres/" property="og:url"/>,
 <meta content="Oppenheimer (2023)" property="og:title"/>,
 <meta content="The story of J. Robert Oppenheimer's role in the development of the atomic bomb during World War II." property="og:description"/>,
 <meta content="https://a.ltrbxd.com/resized/sm/upload/mn/uu/op/02/oppenheimer-2023-1200-1200-675-675-crop-000000.jpg?v=aa1b3a6d85" property="og:image"/>,
 <meta content="1200" property="og:image:width"/>,
 <meta content="675" property="og:image:height"/>,
 <meta content="173683136069040" property="fb:app_id"/>,
 <meta content="summary_large_image" name="twitter:card"/>,

In [56]:
soup_link.find_all("meta", attrs={"name": "twitter:data2"})[0]["content"]

'4.17 out of 5'

In [57]:
float(soup_link.find_all("meta", attrs={"name": "twitter:data2"})[0]["content"].split(' ')[0])

4.17

In [46]:
soup_link.title

<title>‎Oppenheimer (2023) directed by Christopher Nolan • Reviews, film + cast • Letterboxd</title>

In [60]:
for a in soup_link.select('a.text-slug[href^="/films/genre/"]'):
    print(a.text.strip())

Drama
History


In [66]:
soup_link.find_all("meta", attrs={"name": "twitter:data1"})[0]['content']

'Christopher Nolan'

In [123]:
def find_features(soup_genre_link):
    avg_rating = float(soup_genre_link.find_all("meta", attrs={"name": "twitter:data2"})[0]["content"].split(' ')[0])
    genres = [a.text.strip() for a in soup_genre_link.select('a.text-slug[href^="/films/genre/"]')]
    director = soup_genre_link.find_all("meta", attrs={"name": "twitter:data1"})[0]["content"]
    countries = [a.text.strip() for a in soup_genre_link.select('a.text-slug[href^="/films/country/"]')]
    languages = [a.text.strip() for a in soup_genre_link.select('a.text-slug[href^="/films/language/"]')]

    return avg_rating, genres, director, countries, languages

find_features(soup_link)

(4.17,
 ['Drama', 'History'],
 'Christopher Nolan',
 ['UK', 'USA'],
 ['English', 'Dutch', 'English'])